# Lab 4: Autoencoders, VAE, and GAN on MNIST

This notebook covers the full Lab 4 workflow: data preparation, PCA baseline, AE and VAE training, latent-space analysis, generation experiments, and GAN comparison.  
All heavy computations are cache-first with MLflow logging so experiments are not retrained when artifacts already exist.

In [1]:
# !wget https://raw.githubusercontent.com/Lopa10ko/itmo-ml-2025/main/dl/lab-4/requirements.txt
# !wget https://raw.githubusercontent.com/Lopa10ko/itmo-ml-2025/main/dl/lab-4/results_cache.zip

# import zipfile

# with zipfile.ZipFile('results_cache.zip', 'r') as zip_ref:
#     zip_ref.extractall('/content/results_cache')

In [2]:
import sys
import subprocess
from pathlib import Path

is_colab = "google.colab" in sys.modules
req_path = Path("requirements.txt")
if not req_path.exists():
    req_path = Path("dl/lab-4/requirements.txt")

if is_colab and req_path.exists():
    print(f"Colab detected. Installing dependencies from {req_path} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_path), "--quiet"])
else:
    print("Skipping pip install cell (not Colab or requirements.txt not found).")

Skipping pip install cell (not Colab or requirements.txt not found).


In [3]:
import gzip
import hashlib
import json
import pickle
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import mlflow
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import Markdown, display
from mlflow.tracking import MlflowClient
from sklearn.decomposition import PCA
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device}")

BASE_DIR = Path.cwd() / "results_cache"
CACHE_DIR = BASE_DIR / "cache"
DATA_DIR = BASE_DIR / "data"
MLFLOW_TRACKING_URI = (BASE_DIR / "mlruns").resolve()
MLFLOW_TRASH_DIR = MLFLOW_TRACKING_URI / ".trash"
for d in [CACHE_DIR, DATA_DIR, MLFLOW_TRACKING_URI, MLFLOW_TRASH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = str(MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT = "itmo_ml_lab4_vae_gan"
mlflow.set_tracking_uri(f"file:{MLFLOW_TRACKING_URI}")
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = MlflowClient()
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
experiment_id = experiment.experiment_id
print("MLflow URI:", MLFLOW_TRACKING_URI)
print("Experiment ID:", experiment_id)

/Users/lopatenko/Desktop/itmo/itmo-ml-2025/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
MLflow URI: /Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/lab-4/results_cache/mlruns
Experiment ID: 119063023090961984


/Users/lopatenko/Desktop/itmo/itmo-ml-2025/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [4]:
def stable_hash(payload: dict[str, Any]) -> str:
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]


def cache_path(stage: str, cache_key: str) -> Path:
    return CACHE_DIR / f"{stage}_{cache_key}.pkl.gz"


def save_pickle_gz(path: Path, obj: Any) -> None:
    with gzip.open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle_gz(path: Path) -> Any:
    with gzip.open(path, "rb") as f:
        return pickle.load(f)


def cpu_state_dict(state_dict: dict[str, Any]) -> dict[str, Any]:
    return {k: v.detach().cpu().clone() for k, v in state_dict.items()}


def style_fig(fig: go.Figure, title: str, width: int = 950, height: int = 460) -> go.Figure:
    fig.update_layout(template="plotly_white", title=title, width=width, height=height)
    return fig


def to_numpy_images(tensor: torch.Tensor) -> np.ndarray:
    return tensor.detach().cpu().squeeze(1).numpy()


def show_image_grid(images: np.ndarray, title: str, ncols: int = 10, scale: int = 2) -> go.Figure:
    imgs = np.asarray(images)
    if imgs.ndim == 4 and imgs.shape[1] == 1:
        imgs = imgs[:, 0]

    imgs = np.clip(imgs, 0, 1)
    n, h, w = imgs.shape
    nrows = int(np.ceil(n / ncols))

    canvas = np.zeros((nrows * h, ncols * w), dtype=np.float32)
    for i in range(n):
        r, c = divmod(i, ncols)
        canvas[r * h:(r + 1) * h, c * w:(c + 1) * w] = imgs[i]

    fig = px.imshow(canvas, color_continuous_scale="gray", zmin=0.0, zmax=1.0)
    fig.update_layout(
        template="plotly_white",
        title=title,
        margin=dict(l=5, r=5, t=40, b=5),
        height=max(240, nrows * 64 * scale),
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_coloraxes(showscale=False)
    return fig


def find_mlflow_run_id(stage: str, cache_key: str) -> str | None:
    query = f"tags.stage = '{stage}' and tags.cache_key = '{cache_key}'"
    runs = mlflow.search_runs(experiment_ids=[experiment_id], filter_string=query, output_format="list")
    return runs[0].info.run_id if runs else None


def try_restore_cache_from_mlflow(stage: str, cache_key: str, target_path: Path) -> bool:
    run_id = find_mlflow_run_id(stage=stage, cache_key=cache_key)
    if run_id is None:
        return False
    try:
        local_artifact = client.download_artifacts(run_id=run_id, path=f"cache/{target_path.name}")
        target_path.write_bytes(Path(local_artifact).read_bytes())
        return True
    except Exception:
        return False


def train_or_load(
    stage: str,
    cfg: dict[str, Any],
    trainer,
    metric_keys: list[str] | None = None,
) -> tuple[dict[str, Any], Path, bool]:
    metric_keys = metric_keys or []
    key = stable_hash(cfg)
    path = cache_path(stage, key)

    if path.exists():
        try:
            return load_pickle_gz(path), path, True
        except Exception:
            path.unlink(missing_ok=True)

    if try_restore_cache_from_mlflow(stage=stage, cache_key=key, target_path=path):
        try:
            return load_pickle_gz(path), path, True
        except Exception:
            path.unlink(missing_ok=True)

    with mlflow.start_run(run_name=f"{stage}_{key}"):
        mlflow.set_tags({"stage": stage, "cache_key": key})
        mlflow.log_params({k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))})

        t0 = time.perf_counter()
        result = trainer(cfg)
        elapsed = time.perf_counter() - t0
        result["elapsed_sec"] = elapsed

        for m in metric_keys:
            if m in result and isinstance(result[m], (int, float)):
                mlflow.log_metric(m, float(result[m]))

        save_pickle_gz(path, result)
        mlflow.log_artifact(str(path), artifact_path="cache")

    return result, path, False

## Stage 1 - data preparation and EDA

Goal: inspect MNIST distribution, sanity-check image tensors, and test augmentation/noise effects before model training.

In [5]:
base_transform = transforms.ToTensor()

train_ds = datasets.MNIST(root=DATA_DIR, train=True, transform=base_transform, download=True)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, transform=base_transform, download=True)

train_images = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
train_labels = torch.tensor([train_ds[i][1] for i in range(len(train_ds))])
test_images = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
test_labels = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

fig_train_dist = px.bar(
    x=list(range(10)),
    y=np.bincount(train_labels.numpy(), minlength=10),
    labels={"x": "Digit", "y": "Count"},
    title="MNIST train class distribution",
)
style_fig(fig_train_dist, "MNIST train class distribution").show()

sample_idx = np.random.choice(len(train_images), size=20, replace=False)
fig_samples = show_image_grid(train_images[sample_idx].squeeze(1).numpy(), "Random train samples", ncols=10)
fig_samples.show()

In [6]:
summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "images_shape": str(tuple(train_images.shape)),
            "labels_shape": str(tuple(train_labels.shape)),
            "pixel_min": float(train_images.min()),
            "pixel_max": float(train_images.max()),
            "pixel_mean": float(train_images.mean()),
            "pixel_std": float(train_images.std()),
        },
        {
            "split": "test",
            "images_shape": str(tuple(test_images.shape)),
            "labels_shape": str(tuple(test_labels.shape)),
            "pixel_min": float(test_images.min()),
            "pixel_max": float(test_images.max()),
            "pixel_mean": float(test_images.mean()),
            "pixel_std": float(test_images.std()),
        },
    ]
)
summary_df

,split,images_shape,labels_shape,pixel_min,pixel_max,pixel_mean,pixel_std
0,train,"(60000, 1, 28, 28)","(60000,)",0.0,1.0,0.130660,0.308108
1,test,"(10000, 1, 28, 28)","(10000,)",0.0,1.0,0.132515,0.310480


In [7]:
augment_transform = transforms.Compose([
    transforms.RandomAffine(degrees=20, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    transforms.ToTensor(),
])
aug_ds = datasets.MNIST(root=DATA_DIR, train=True, transform=augment_transform, download=True)

aug_images = torch.stack([aug_ds[i][0] for i in sample_idx])
fig_aug = show_image_grid(aug_images.squeeze(1).numpy(), "Augmented samples", ncols=10)
fig_aug.show()

noise_std = 0.25
noisy_batch = torch.clamp(train_images[sample_idx] + noise_std * torch.randn_like(train_images[sample_idx]), 0, 1)
fig_noisy = show_image_grid(noisy_batch.squeeze(1).numpy(), "Noisy samples (gaussian noise)", ncols=10)
fig_noisy.show()

### Stage 1 insights (data preparation)

**Observed:** Train and test tensors are in `[0,1]`, class counts are close to balanced, and augmentation/noise produce still-recognizable digits.  
**Interpretation:** The data pipeline is numerically stable for BCE-based reconstruction training, and class imbalance is unlikely to dominate optimization.  
**Answers to questions:**
- Class distribution should not create strong prior bias toward one digit class.
- Rotation/translation augmentation is plausible and useful for invariance.
- Gaussian noise increases reconstruction difficulty, motivating denoising experiments.

## Stage 2 - PCA baseline

We fit PCA on flattened images, inspect explained variance, and compare reconstruction quality against neural approaches later.

In [8]:
pca_train_ds = datasets.MNIST(root=DATA_DIR, train=True, transform=base_transform, download=True)
pca_test_ds = datasets.MNIST(root=DATA_DIR, train=False, transform=base_transform, download=True)
pca_train_images = torch.stack([pca_train_ds[i][0] for i in range(len(pca_train_ds))])
pca_train_labels = torch.tensor([pca_train_ds[i][1] for i in range(len(pca_train_ds))])
pca_test_images = torch.stack([pca_test_ds[i][0] for i in range(len(pca_test_ds))])
pca_test_labels = torch.tensor([pca_test_ds[i][1] for i in range(len(pca_test_ds))])

train_flat = pca_train_images.view(len(pca_train_images), -1).numpy()
test_flat = pca_test_images.view(len(pca_test_images), -1).numpy()

pca_components = 32
pca = PCA(n_components=pca_components, random_state=SEED)
pca.fit(train_flat)

test_proj = pca.transform(test_flat)
test_recon = pca.inverse_transform(test_proj)
eps = 1e-8
pca_bce = float(-np.mean(
    np.clip(test_flat, 0, 1) * np.log(np.clip(test_recon, eps, 1 - eps))
    + (1 - np.clip(test_flat, 0, 1)) * np.log(np.clip(1 - test_recon, eps, 1 - eps))
))
print("PCA BCE:", pca_bce)

cum_var = np.cumsum(pca.explained_variance_ratio_)
fig_var = px.line(
    x=np.arange(1, pca_components + 1),
    y=cum_var,
    labels={"x": "Components", "y": "Cumulative explained variance"},
    title="PCA explained variance",
)
style_fig(fig_var, "PCA cumulative explained variance").show()

pca_recon_images = np.clip(test_recon.reshape(-1, 28, 28), 0, 1)
fig_pca_recon = show_image_grid(pca_recon_images[:20], "PCA reconstructions (first 20 test images)", ncols=10)
fig_pca_recon.show()

pca3 = PCA(n_components=3, random_state=SEED)
latent3 = pca3.fit_transform(train_flat[:8000])
fig_pca_latent = px.scatter_3d(
    x=latent3[:, 0],
    y=latent3[:, 1],
    z=latent3[:, 2],
    color=pca_train_labels[:8000].numpy().astype(str),
    opacity=0.7,
    title="PCA 3D latent space",
)
fig_pca_latent.update_layout(template="plotly_white", width=950, height=650)
fig_pca_latent.show()

PCA BCE: 0.11802664399147034


In [9]:
display(Markdown(
    f"""
### Stage 2 insights (PCA)

**Observed:** PCA reconstruction BCE is `{pca_bce:.4f}`; cumulative explained variance at 32 components is `{float(cum_var[-1]):.4f}`.  
**Interpretation:** Even high retained variance does not fully capture nonlinear digit structure, so reconstructions stay smoother than neural methods.  
**Answers to questions:**
- PCA provides a clear linear baseline but underfits curved manifolds.
- Explained variance is useful for compression choice, not a full proxy for perceptual reconstruction quality.
"""
))


### Stage 2 insights (PCA)

**Observed:** PCA reconstruction BCE is `0.1180`; cumulative explained variance at 32 components is `0.7436`.  
**Interpretation:** Even high retained variance does not fully capture nonlinear digit structure, so reconstructions stay smoother than neural methods.  
**Answers to questions:**
- PCA provides a clear linear baseline but underfits curved manifolds.
- Explained variance is useful for compression choice, not a full proxy for perceptual reconstruction quality.


## Stage 3 - Autoencoder (AE)

An autoencoder learns two mappings: encoder $f_\theta(x)=z$ and decoder $g_\phi(z)=\hat{x}$.
The model is trained to minimize reconstruction error between input and output.

$$
z = f_\theta(x), \quad \hat{x} = g_\phi(z)
$$

$$
\mathcal{L}_{AE}(\theta,\phi)=\frac{1}{N}\sum_{i=1}^{N}\ell\left(x_i,\hat{x}_i\right)
$$

In this notebook, for AE we use BCE as the reconstruction criterion:

$$
\ell_{BCE}(x,\hat{x})=-\sum_{p}\left[x_p\log(\hat{x}_p)+(1-x_p)\log(1-\hat{x}_p)\right]
$$

Train a compact convolutional autoencoder with cache-first execution and MLflow logging.

In [10]:
train_ds = datasets.MNIST(root=DATA_DIR, train=True, transform=augment_transform, download=True)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, transform=augment_transform, download=True)
train_images = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
train_labels = torch.tensor([train_ds[i][1] for i in range(len(train_ds))])
test_images = torch.stack([test_ds[i][0] for i in range(len(test_ds))])
test_labels = torch.tensor([test_ds[i][1] for i in range(len(test_ds))])

In [11]:
@dataclass
class TrainConfig:
    batch_size: int = 256
    epochs: int = 8
    lr: float = 1e-3
    latent_dim: int = 16
    beta: float = 1.0


class ConvAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


class ConvVAE(nn.Module):
    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu = nn.Linear(32 * 7 * 7, latent_dim)
        self.fc_logvar = nn.Linear(32 * 7 * 7, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, 32 * 7 * 7)
        self.dec = nn.Sequential(
            nn.Unflatten(1, (32, 7, 7)),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.dec(self.fc_dec(z))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def make_loaders(batch_size: int, train_limit: int | None = 20000, test_limit: int | None = 5000):
    train_idx = np.arange(len(train_ds)) if train_limit is None else np.arange(train_limit)
    test_idx = np.arange(len(test_ds)) if test_limit is None else np.arange(test_limit)
    train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(Subset(test_ds, test_idx), batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, test_loader

In [12]:
from itertools import product

HEAVY_GRID_CONFIGS = {
    "ae": {
        "grid": {
            "epochs": [40, 60],
            "lr": [1e-3, 5e-4],
            "latent_dim": [16, 32],
            "batch_size": [256],
            "beta": [1.0],
        },
        "metric_key": "test_bce",
        "direction": "min",
        "stage": "ae_grid",
    },
    "vae": {
        "grid": {
            "epochs": [40, 60],
            "lr": [1e-3, 5e-4],
            "latent_dim": [16, 32],
            "batch_size": [256],
            "beta": [0.5, 1.0, 2.0],
        },
        "metric_key": "test_total",
        "direction": "min",
        "stage": "vae_grid",
    },
    "gan": {
        "grid": {
            "epochs": [50, 70],
            "lr": [2e-4, 1e-4],
            "z_dim": [64, 96],
            "batch_size": [256],
        },
        "metric_key": "g_last",
        "direction": "min",
        "stage": "gan_grid",
    },
    "dae": {
        "grid": {
            "epochs": [40, 60],
            "lr": [1e-3, 5e-4],
            "latent_dim": [16, 32],
            "batch_size": [256],
            "beta": [1.0],
            "noise_std": [0.2, 0.35],
        },
        "metric_key": "test_bce",
        "direction": "min",
        "stage": "dae_grid",
    },
    "dvae": {
        "grid": {
            "epochs": [40, 60],
            "lr": [1e-3, 5e-4],
            "latent_dim": [16, 32],
            "batch_size": [256],
            "beta": [0.5, 1.0, 2.0],
            "noise_std": [0.2, 0.35],
        },
        "metric_key": "test_total",
        "direction": "min",
        "stage": "dvae_grid",
    },
}


def expand_grid(grid: dict[str, list[Any]]) -> list[dict[str, Any]]:
    keys = list(grid.keys())
    return [dict(zip(keys, vals)) for vals in product(*(grid[k] for k in keys))]


def run_grid_search(
    name: str,
    spec: dict[str, Any],
    trainer,
) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    rows = []
    best_res = None
    best_cfg = None
    best_metric = None

    all_cfgs = expand_grid(spec["grid"])
    cfg_iter = tqdm(all_cfgs, desc=f"{name} grid", leave=True)

    for cfg in cfg_iter:
        res, cache_p, loaded = train_or_load(
            stage=spec["stage"],
            cfg=cfg,
            trainer=trainer,
            metric_keys=[spec["metric_key"]],
        )
        metric = float(res[spec["metric_key"]])
        cfg_iter.set_postfix({"best_so_far": f"{metric:.4f}", "loaded": loaded})
        rows.append({
            "model": name,
            "metric": metric,
            "metric_key": spec["metric_key"],
            "cache_path": str(cache_p),
            "loaded": loaded,
            **cfg,
        })

        if best_metric is None:
            best_metric, best_cfg, best_res = metric, cfg, res
        elif spec["direction"] == "min" and metric < best_metric:
            best_metric, best_cfg, best_res = metric, cfg, res
        elif spec["direction"] == "max" and metric > best_metric:
            best_metric, best_cfg, best_res = metric, cfg, res

    df = pd.DataFrame(rows).sort_values("metric", ascending=(spec["direction"] == "min")).reset_index(drop=True)
    return df, best_cfg, best_res

In [13]:
def train_ae(cfg: dict[str, Any]) -> dict[str, Any]:
    tc = TrainConfig(**cfg)
    train_loader, test_loader = make_loaders(tc.batch_size)
    model = ConvAE(latent_dim=tc.latent_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=tc.lr)

    train_losses, val_losses = [], []
    for _ in tqdm(range(tc.epochs), desc="AE epochs", leave=False):
        model.train()
        tr_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            opt.zero_grad()
            x_hat, _ = model(x)
            loss = F.binary_cross_entropy(x_hat, x)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * x.size(0)
        train_losses.append(tr_loss / len(train_loader.dataset))

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, _ = model(x)
                vl_loss += F.binary_cross_entropy(x_hat, x).item() * x.size(0)
        val_losses.append(vl_loss / len(test_loader.dataset))

    model.eval()
    with torch.no_grad():
        x_batch, y_batch = next(iter(test_loader))
        x_batch = x_batch.to(device)
        x_hat, z = model(x_batch)
    result = {
        "cfg": cfg,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_bce": float(val_losses[-1]),
        "orig": x_batch.detach().cpu().numpy(),
        "recon": x_hat.detach().cpu().numpy(),
        "labels": y_batch.numpy(),
        "latent": z.detach().cpu().numpy(),
        "model_state": cpu_state_dict(model.state_dict()),
    }
    return result


def train_vae(cfg: dict[str, Any]) -> dict[str, Any]:
    tc = TrainConfig(**cfg)
    train_loader, test_loader = make_loaders(tc.batch_size)
    model = ConvVAE(latent_dim=tc.latent_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=tc.lr)

    history = {"train_total": [], "train_rec": [], "train_kl": [], "val_total": []}
    for _ in tqdm(range(tc.epochs), desc="VAE epochs", leave=False):
        model.train()
        t_total = t_rec = t_kl = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            opt.zero_grad()
            x_hat, mu, logvar, _ = model(x)
            rec = F.binary_cross_entropy(x_hat, x, reduction="sum") / x.size(0)
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
            loss = rec + tc.beta * kl
            loss.backward()
            opt.step()
            t_total += loss.item() * x.size(0)
            t_rec += rec.item() * x.size(0)
            t_kl += kl.item() * x.size(0)

        history["train_total"].append(t_total / len(train_loader.dataset))
        history["train_rec"].append(t_rec / len(train_loader.dataset))
        history["train_kl"].append(t_kl / len(train_loader.dataset))

        model.eval()
        v_total = 0.0
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(device)
                x_hat, mu, logvar, _ = model(x)
                rec = F.binary_cross_entropy(x_hat, x, reduction="sum") / x.size(0)
                kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
                v_total += (rec + tc.beta * kl).item() * x.size(0)
        history["val_total"].append(v_total / len(test_loader.dataset))

    model.eval()
    with torch.no_grad():
        x_batch, y_batch = next(iter(test_loader))
        x_batch = x_batch.to(device)
        x_hat, mu, logvar, z = model(x_batch)

    return {
        "cfg": cfg,
        "history": history,
        "test_total": float(history["val_total"][-1]),
        "orig": x_batch.detach().cpu().numpy(),
        "recon": x_hat.detach().cpu().numpy(),
        "labels": y_batch.numpy(),
        "mu": mu.detach().cpu().numpy(),
        "logvar": logvar.detach().cpu().numpy(),
        "latent": z.detach().cpu().numpy(),
        "model_state": cpu_state_dict(model.state_dict()),
    }

In [14]:
ae_grid_df, ae_best_cfg, ae_res = run_grid_search("AE", HEAVY_GRID_CONFIGS["ae"], trainer=train_ae)
display(ae_grid_df.head(10))
print("Best AE config:", ae_best_cfg)

fig_ae_loss = go.Figure()
fig_ae_loss.add_trace(go.Scatter(y=ae_res["train_losses"], mode="lines+markers", name="train_bce"))
fig_ae_loss.add_trace(go.Scatter(y=ae_res["val_losses"], mode="lines+markers", name="val_bce"))
style_fig(fig_ae_loss, "AE training curves (best grid config)", height=420).show()

orig = ae_res["orig"].squeeze(1)[:20]
recon = ae_res["recon"].squeeze(1)[:20]
stack = np.concatenate([orig, recon], axis=0)
fig_ae_recon = show_image_grid(stack, "AE best config: originals + reconstructions", ncols=10)
fig_ae_recon.show()

AE grid: 100%|██████████| 8/8 [00:00<00:00, 61.22it/s, best_so_far=0.0930, loaded=1]


,model,metric,metric_key,cache_path,loaded,epochs,lr,latent_dim,batch_size,beta
0,AE,0.089783,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,1.0
1,AE,0.091051,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,1.0
2,AE,0.092971,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,1.0
3,AE,0.093866,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,32,256,1.0
4,AE,0.110579,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,16,256,1.0
5,AE,0.113135,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,16,256,1.0
6,AE,0.113333,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,16,256,1.0
7,AE,0.117723,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,16,256,1.0


Best AE config: {'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 1.0}


In [15]:
ae_train_last = float(ae_res['train_losses'][-1])
ae_val_last = float(ae_res['val_losses'][-1])
display(Markdown(
    f"""
### Stage 3 insights (AE)

**Observed:** Best AE grid config is `{ae_best_cfg}` with final train BCE `{ae_train_last:.4f}` and val BCE `{ae_val_last:.4f}`.  
**Interpretation:** Longer training and tuned latent dimension/lr improve deterministic reconstruction stability.  
**Answers to questions:**
- AE reconstructs most digits reliably, with occasional smoothing of thin strokes.
- BCE trends support convergence and provide a strong deterministic baseline for VAE.
"""
))


### Stage 3 insights (AE)

**Observed:** Best AE grid config is `{'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 1.0}` with final train BCE `0.0906` and val BCE `0.0898`.  
**Interpretation:** Longer training and tuned latent dimension/lr improve deterministic reconstruction stability.  
**Answers to questions:**
- AE reconstructs most digits reliably, with occasional smoothing of thin strokes.
- BCE trends support convergence and provide a strong deterministic baseline for VAE.


## Stage 4 - Variational Autoencoder (VAE)

VAE assumes a latent variable model with prior $p(z)=\mathcal{N}(0,I)$ and decoder likelihood $p_\phi(x\mid z)$.
The encoder outputs parameters of the approximate posterior $q_\theta(z\mid x)=\mathcal{N}(\mu(x),\sigma^2(x))$.

Reparameterization trick:

$$
z = \mu + \sigma \odot \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,I)
$$

The objective is the negative ELBO, written as reconstruction term plus KL regularization:

$$
\mathcal{L}_{VAE}=\underbrace{\mathcal{L}_{rec}}_{\text{BCE-based}} + \beta\underbrace{D_{KL}\left(q_\theta(z\mid x)\,\|\,p(z)\right)}_{\text{latent regularization}}
$$

For diagonal Gaussian posterior, KL has closed form:

$$
D_{KL}= -\frac{1}{2}\sum_j \left(1+\log\sigma_j^2-\mu_j^2-\sigma_j^2\right)
$$

The VAE adds a probabilistic latent space and KL regularization, enabling smoother generation and interpolation.

In [16]:
ae_latent = ae_res['latent']
if ae_latent.shape[1] > 2:
    ae_latent_2d = PCA(n_components=2, random_state=SEED).fit_transform(ae_latent)
else:
    ae_latent_2d = ae_latent

fig_ae_latent = px.scatter(
    x=ae_latent_2d[:, 0],
    y=ae_latent_2d[:, 1],
    color=ae_res['labels'].astype(str),
    title='AE latent space (projected to 2D)',
)
style_fig(fig_ae_latent, 'AE latent space (2D)').show()

In [17]:
ae_model_eval = ConvAE(latent_dim=int(ae_res['cfg']['latent_dim'])).to(device)
ae_model_eval.load_state_dict(ae_res['model_state'])
ae_model_eval.eval()

orig_batch = ae_res['orig']
orig_flat = orig_batch.reshape(orig_batch.shape[0], -1)
pca_pre_flat = np.clip(pca.inverse_transform(pca.transform(orig_flat)), 0, 1)
pca_pre_batch = pca_pre_flat.reshape(orig_batch.shape)

with torch.no_grad():
    x_in = torch.tensor(pca_pre_batch, dtype=torch.float32, device=device)
    x_hat_pre, _ = ae_model_eval(x_in)

eps = 1e-8
base_bce = float(-np.mean(np.clip(orig_batch,0,1)*np.log(np.clip(ae_res['recon'],eps,1-eps)) + (1-np.clip(orig_batch,0,1))*np.log(np.clip(1-ae_res['recon'],eps,1-eps))))
pre_bce = float(-np.mean(np.clip(orig_batch,0,1)*np.log(np.clip(x_hat_pre.cpu().numpy(),eps,1-eps)) + (1-np.clip(orig_batch,0,1))*np.log(np.clip(1-x_hat_pre.cpu().numpy(),eps,1-eps))))

pca_pre_df = pd.DataFrame([
    {'setting': 'AE direct input', 'bce_recon': base_bce},
    {'setting': 'PCA -> AE input', 'bce_recon': pre_bce},
])
fig_pca_pre = px.bar(pca_pre_df, x='setting', y='bce_recon', color='setting', title='Does PCA preprocessing help AE reconstruction?')
style_fig(fig_pca_pre, 'PCA preprocessing before AE').show()

display(Markdown(
    f"""
### Additional comparison: PCA as preprocessing before AE

**Observed:** BCE for direct AE input is `{base_bce:.4f}`, while PCA-preprocessed AE input gives `{pre_bce:.4f}`.  
**Answer:** This provides direct evidence whether PCA preprocessing helps or hurts AE in this setup.
"""
))


### Additional comparison: PCA as preprocessing before AE

**Observed:** BCE for direct AE input is `0.0858`, while PCA-preprocessed AE input gives `0.1647`.  
**Answer:** This provides direct evidence whether PCA preprocessing helps or hurts AE in this setup.


In [18]:
vae_grid_df, vae_best_cfg, vae_res = run_grid_search("VAE", HEAVY_GRID_CONFIGS["vae"], trainer=train_vae)
display(vae_grid_df.head(10))
print("Best VAE config:", vae_best_cfg)

hist = vae_res["history"]
fig_vae_loss = go.Figure()
fig_vae_loss.add_trace(go.Scatter(y=hist["train_total"], mode="lines+markers", name="train_total"))
fig_vae_loss.add_trace(go.Scatter(y=hist["train_rec"], mode="lines+markers", name="train_reconstruction"))
fig_vae_loss.add_trace(go.Scatter(y=hist["train_kl"], mode="lines+markers", name="train_kl"))
fig_vae_loss.add_trace(go.Scatter(y=hist["val_total"], mode="lines+markers", name="val_total"))
style_fig(fig_vae_loss, "VAE loss decomposition (best grid config)", height=460).show()

orig_v = vae_res["orig"].squeeze(1)[:20]
recon_v = vae_res["recon"].squeeze(1)[:20]
fig_vae_recon = show_image_grid(np.concatenate([orig_v, recon_v], axis=0), "VAE best config reconstructions", ncols=10)
fig_vae_recon.show()

latent_for_plot = vae_res["mu"]
if latent_for_plot.shape[1] > 2:
    latent_for_plot = PCA(n_components=2, random_state=SEED).fit_transform(latent_for_plot)

fig_vae_latent = px.scatter(
    x=latent_for_plot[:, 0],
    y=latent_for_plot[:, 1],
    color=vae_res["labels"].astype(str),
    title="VAE latent space (projected to 2D)",
)
style_fig(fig_vae_latent, "VAE latent space (2D)").show()

VAE grid: 100%|██████████| 24/24 [00:00<00:00, 74.88it/s, best_so_far=146.1843, loaded=1]


,model,metric,metric_key,cache_path,loaded,epochs,lr,latent_dim,batch_size,beta
0,VAE,101.446786,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,0.5
1,VAE,103.359799,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,0.5
2,VAE,103.661708,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,0.5
3,VAE,105.491698,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,32,256,0.5
4,VAE,108.309797,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,16,256,0.5
5,VAE,110.562137,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,16,256,0.5
6,VAE,112.447365,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,16,256,0.5
7,VAE,115.094001,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,16,256,0.5
8,VAE,121.102721,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,1.0
9,VAE,121.280424,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,1.0


Best VAE config: {'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 0.5}


In [19]:
v_hist = vae_res['history']
vae_total = float(v_hist['val_total'][-1])
vae_rec = float(v_hist['train_rec'][-1])
vae_kl = float(v_hist['train_kl'][-1])
display(Markdown(
    f"""
### Stage 4 insights (VAE)

**Observed:** Best VAE grid config is `{vae_best_cfg}` with validation objective `{vae_total:.4f}`, reconstruction `{vae_rec:.4f}`, and KL `{vae_kl:.4f}`.  
**Interpretation:** KL remains active under tuned `beta`, giving regularized latent geometry with competitive reconstruction.  
**Answers to questions:**
- The reconstruction-vs-KL balance is visible and controlled by hyperparameter search.
- Latent structure is regularized rather than purely memorized.
"""
))


### Stage 4 insights (VAE)

**Observed:** Best VAE grid config is `{'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 0.5}` with validation objective `101.4468`, reconstruction `80.8286`, and KL `44.4360`.  
**Interpretation:** KL remains active under tuned `beta`, giving regularized latent geometry with competitive reconstruction.  
**Answers to questions:**
- The reconstruction-vs-KL balance is visible and controlled by hyperparameter search.
- Latent structure is regularized rather than purely memorized.


## Stage 5 - PCA vs AE vs VAE comparison

In [20]:
eps = 1e-8
def bce_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.clip(y_true, 0.0, 1.0)
    y_pred = np.clip(y_pred, eps, 1.0 - eps)
    return float(-np.mean(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred)))

base_batch = ae_res["orig"]
ae_batch_recon = ae_res["recon"]
vae_batch_recon = vae_res["recon"]
base_batch_flat = base_batch.reshape(base_batch.shape[0], -1)
pca_batch_recon = np.clip(pca.inverse_transform(pca.transform(base_batch_flat)).reshape(base_batch.shape), 0, 1)

comparison_df = pd.DataFrame(
    [
        {"model": "PCA", "bce_recon": bce_np(base_batch, pca_batch_recon)},
        {"model": "AE", "bce_recon": bce_np(base_batch, ae_batch_recon)},
        {"model": "VAE", "bce_recon": bce_np(base_batch, vae_batch_recon)},
    ]
)
fig_cmp = px.bar(comparison_df, x="model", y="bce_recon", color="model", title="Reconstruction BCE comparison")
style_fig(fig_cmp, "PCA vs AE vs VAE reconstruction BCE").show()

show_idx = np.arange(10)
orig_show = test_images[show_idx].squeeze(1).numpy()
pca_show = pca_batch_recon.squeeze(1)[show_idx]
ae_show = ae_res["recon"].squeeze(1)[:10]
vae_show = vae_res["recon"].squeeze(1)[:10]
all_show = np.concatenate([orig_show, pca_show, ae_show, vae_show], axis=0)
fig_rows = show_image_grid(all_show, "Rows: original / PCA / AE / VAE", ncols=10)
fig_rows.show()

/var/folders/4c/lkr5h1ws6b7ffd_1cx_hw_g80000gn/T/ipykernel_41206/1670914493.py:5: RuntimeWarning: divide by zero encountered in log
  return float(-np.mean(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred)))
/var/folders/4c/lkr5h1ws6b7ffd_1cx_hw_g80000gn/T/ipykernel_41206/1670914493.py:5: RuntimeWarning: invalid value encountered in multiply
  return float(-np.mean(y_true * np.log(y_pred) + (1.0 - y_true) * np.log(1.0 - y_pred)))


Control questions (Stage 4-5): PCA is linear and often misses curved manifolds, so it can lose details in strokes and digit thickness. AE/VAE capture nonlinear structure and usually reconstruct better. PCA can still be useful as a fast baseline, compression tool, or preprocessing step when low compute and interpretability are priorities.

In [21]:
cmp_sorted = comparison_df.sort_values('bce_recon')
best_model = cmp_sorted.iloc[0]['model']
best_score = float(cmp_sorted.iloc[0]['bce_recon'])
display(Markdown(
    f"""
### Stage 5 insights (PCA vs AE vs VAE)

**Observed:** Best reconstruction BCE on the shared comparison batch is **{best_model}** with `{best_score:.4f}`.  
**Interpretation:** Nonlinear models better preserve digit structure than PCA, while PCA keeps interpretability and fast compression.  
**Answers to questions:**
- The shared BCE metric gives a direct quantitative ranking.
- The latent plots (PCA, AE, VAE) show how linear vs nonlinear embeddings organize class structure differently.
"""
))


### Stage 5 insights (PCA vs AE vs VAE)

**Observed:** Best reconstruction BCE on the shared comparison batch is **AE** with `0.0858`.  
**Interpretation:** Nonlinear models better preserve digit structure than PCA, while PCA keeps interpretability and fast compression.  
**Answers to questions:**
- The shared BCE metric gives a direct quantitative ranking.
- The latent plots (PCA, AE, VAE) show how linear vs nonlinear embeddings organize class structure differently.


## Stage 6 - VAE generation studies

Experiments: random sampling, interpolation, latent grid traversal, noise sensitivity, and latent-dimension sweep.

In [22]:
def restore_vae_from_result(result: dict[str, Any]) -> ConvVAE:
    latent_dim = int(result["cfg"]["latent_dim"])
    model = ConvVAE(latent_dim=latent_dim).to(device)
    model.load_state_dict(result["model_state"])
    model.eval()
    return model

vae_model = restore_vae_from_result(vae_res)
latent_dim = int(vae_res["cfg"]["latent_dim"])

with torch.no_grad():
    z_rand = torch.randn(20, latent_dim, device=device)
    gen_rand = vae_model.decode(z_rand).cpu().numpy().squeeze(1)
fig_rand = show_image_grid(gen_rand, "VAE random samples", ncols=10)
fig_rand.show()

with torch.no_grad():
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    alphas = torch.linspace(0, 1, steps=12, device=device).unsqueeze(1)
    z_interp = (1 - alphas) * z0 + alphas * z1
    interp_imgs = vae_model.decode(z_interp).cpu().numpy().squeeze(1)
fig_interp = show_image_grid(interp_imgs, "VAE interpolation", ncols=12)
fig_interp.show()

if latent_dim >= 2:
    grid = np.linspace(-2.5, 2.5, 10)
    points = np.array([[x, y] + [0] * (latent_dim - 2) for y in grid for x in grid], dtype=np.float32)
    with torch.no_grad():
        grid_imgs = vae_model.decode(torch.tensor(points, device=device)).cpu().numpy().squeeze(1)
    fig_grid = show_image_grid(grid_imgs, "VAE latent 2D grid traversal", ncols=10, scale=1)
    fig_grid.show()

noise_levels = [0.0, 0.2, 0.5, 1.0]
noise_rows = []
with torch.no_grad():
    base_z = torch.randn(10, latent_dim, device=device)
    for n in noise_levels:
        noisy = base_z + n * torch.randn_like(base_z)
        imgs = vae_model.decode(noisy).cpu().numpy().squeeze(1)
        noise_rows.append(imgs)
fig_noise = show_image_grid(np.concatenate(noise_rows, axis=0), "Noise sensitivity (rows by increasing noise)", ncols=10)
fig_noise.show()

In [23]:
latent_dim_sweep = [2, 8, 16, 32]
ld_results = []
for ld in latent_dim_sweep:
    cfg = {
        "epochs": 30,
        "latent_dim": ld,
        "lr": float(vae_best_cfg["lr"]),
        "batch_size": int(vae_best_cfg["batch_size"]),
        "beta": float(vae_best_cfg["beta"]),
    }
    res, _, _ = train_or_load(stage="vae_latent_sweep", cfg=cfg, trainer=train_vae, metric_keys=["test_total"])
    ld_results.append({"latent_dim": ld, "metric": float(res["test_total"])})

ld_df = pd.DataFrame(ld_results)
fig_ld = px.line(ld_df, x="latent_dim", y="metric", markers=True, title="VAE latent dimension sweep")
style_fig(fig_ld, "VAE latent_dim impact on validation objective").show()

## Stage 7 - GAN

A simple MLP GAN baseline is trained with the same cache-first policy and compared against VAE generation.

In [24]:
ld_best = ld_df.sort_values('metric').iloc[0]
display(Markdown(
    f"""
### Stage 6 insights (generation and latent sweeps)

**Observed:** Using VAE base config `{vae_best_cfg}`, latent-dim sweep is best at `latent_dim={int(ld_best['latent_dim'])}` with objective `{float(ld_best['metric']):.4f}`.  
**Interpretation:** Too-small latent spaces underfit diversity, while too-large spaces can weaken regularization benefits.  
**Answers to questions:**
- Random samples are mostly digit-like due to regularized latent prior.
- Noise robustness is limited: quality drops as latent perturbation increases.
- Latent dimension controls the fidelity/diversity trade-off.
"""
))


### Stage 6 insights (generation and latent sweeps)

**Observed:** Using VAE base config `{'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 0.5}`, latent-dim sweep is best at `latent_dim=32` with objective `104.2820`.  
**Interpretation:** Too-small latent spaces underfit diversity, while too-large spaces can weaken regularization benefits.  
**Answers to questions:**
- Random samples are mostly digit-like due to regularized latent prior.
- Noise robustness is limited: quality drops as latent perturbation increases.
- Latent dimension controls the fidelity/diversity trade-off.


In [25]:
class GanGenerator(nn.Module):
    def __init__(self, z_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class GanDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_gan(cfg: dict[str, Any]) -> dict[str, Any]:
    z_dim = int(cfg.get("z_dim", 64))
    epochs = int(cfg.get("epochs", 8))
    lr = float(cfg.get("lr", 2e-4))
    batch_size = int(cfg.get("batch_size", 256))

    train_loader, _ = make_loaders(batch_size=batch_size, train_limit=20000, test_limit=1000)
    G = GanGenerator(z_dim=z_dim).to(device)
    D = GanDiscriminator().to(device)

    opt_g = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCELoss()

    g_losses, d_losses = [], []
    for _ in tqdm(range(epochs), desc="GAN epochs", leave=False):
        g_ep, d_ep = 0.0, 0.0
        for x, _ in train_loader:
            x = x.to(device)
            bs = x.size(0)
            real = torch.ones(bs, 1, device=device)
            fake = torch.zeros(bs, 1, device=device)

            opt_d.zero_grad()
            z = torch.randn(bs, z_dim, device=device)
            x_fake = G(z).detach()
            loss_d = bce(D(x), real) + bce(D(x_fake), fake)
            loss_d.backward()
            opt_d.step()

            opt_g.zero_grad()
            z = torch.randn(bs, z_dim, device=device)
            x_fake = G(z)
            loss_g = bce(D(x_fake), real)
            loss_g.backward()
            opt_g.step()

            g_ep += loss_g.item() * bs
            d_ep += loss_d.item() * bs

        g_losses.append(g_ep / len(train_loader.dataset))
        d_losses.append(d_ep / len(train_loader.dataset))

    with torch.no_grad():
        z = torch.randn(20, z_dim, device=device)
        samples = G(z).cpu().numpy()

    return {
        "cfg": cfg,
        "g_losses": g_losses,
        "d_losses": d_losses,
        "g_last": float(g_losses[-1]),
        "d_last": float(d_losses[-1]),
        "samples": samples,
        "G_state": cpu_state_dict(G.state_dict()),
        "D_state": cpu_state_dict(D.state_dict()),
    }

In [26]:
gan_grid_df, gan_best_cfg, gan_res = run_grid_search("GAN", HEAVY_GRID_CONFIGS["gan"], trainer=train_gan)
display(gan_grid_df.head(10))
print("Best GAN config:", gan_best_cfg)

fig_gan_losses = go.Figure()
fig_gan_losses.add_trace(go.Scatter(y=gan_res["g_losses"], mode="lines+markers", name="G loss"))
fig_gan_losses.add_trace(go.Scatter(y=gan_res["d_losses"], mode="lines+markers", name="D loss"))
style_fig(fig_gan_losses, "GAN training dynamics (best grid config)", height=420).show()

fig_gan_samples = show_image_grid(gan_res["samples"].squeeze(1), "GAN random samples", ncols=10)
fig_gan_samples.show()

fig_vae_vs_gan = show_image_grid(
    np.concatenate([gen_rand[:20], gan_res["samples"].squeeze(1)[:20]], axis=0),
    "VAE samples (row 1-2) vs GAN samples (row 3-4)",
    ncols=10,
)
fig_vae_vs_gan.show()

GAN grid: 100%|██████████| 8/8 [00:00<00:00, 35.69it/s, best_so_far=10.3749, loaded=1]


,model,metric,metric_key,cache_path,loaded,epochs,lr,z_dim,batch_size
0,GAN,6.417739,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,70,0.0002,64,256
1,GAN,8.095298,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,50,0.0001,96,256
2,GAN,8.560804,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,50,0.0001,64,256
3,GAN,10.374891,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,70,0.0001,96,256
4,GAN,11.482614,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,70,0.0001,64,256
5,GAN,36.796905,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,50,0.0002,64,256
6,GAN,37.015421,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,70,0.0002,96,256
7,GAN,49.246741,g_last,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,50,0.0002,96,256


Best GAN config: {'epochs': 70, 'lr': 0.0002, 'z_dim': 64, 'batch_size': 256}


In [27]:
G = GanGenerator(z_dim=int(gan_res["cfg"]["z_dim"]))
G.load_state_dict(gan_res["G_state"])
G = G.to(device).eval()

with torch.no_grad():
    z0 = torch.randn(1, int(gan_res["cfg"]["z_dim"]), device=device)
    z1 = torch.randn(1, int(gan_res["cfg"]["z_dim"]), device=device)
    alphas = torch.linspace(0, 1, steps=12, device=device).unsqueeze(1)
    z_interp_gan = (1 - alphas) * z0 + alphas * z1
    gan_interp = G(z_interp_gan).cpu().numpy().squeeze(1)
fig_gan_interp = show_image_grid(gan_interp, "GAN latent interpolation", ncols=12)
fig_gan_interp.show()

Control questions (Stage 6): interpolation is usually smooth when latent space is well-regularized; abrupt artifacts indicate weak manifold continuity. Increasing latent noise gradually degrades shape consistency. Very small latent dimensions underfit class-specific details, while very large dimensions can reduce regularization benefits and produce less stable generations.

In [28]:
g_last = float(gan_res['g_losses'][-1])
d_last = float(gan_res['d_losses'][-1])
display(Markdown(
    f"""
### Stage 7 insights (GAN vs VAE)

**Observed:** Best GAN grid config is `{gan_best_cfg}` with final losses `G={g_last:.4f}` and `D={d_last:.4f}`; sample sharpness improves but stability still varies.  
**Interpretation:** Grid tuning reduces instability but GAN remains more optimization-sensitive than VAE.  
**Answers to questions:**
- VAE provides smoother, more controllable latent interpolation.
- GAN may not converge to a stable solution (requires more tuning or careful hyperparameter selection and implementation).
"""
))


### Stage 7 insights (GAN vs VAE)

**Observed:** Best GAN grid config is `{'epochs': 70, 'lr': 0.0002, 'z_dim': 64, 'batch_size': 256}` with final losses `G=6.4177` and `D=0.0287`; sample sharpness improves but stability still varies.  
**Interpretation:** Grid tuning reduces instability but GAN remains more optimization-sensitive than VAE.  
**Answers to questions:**
- VAE provides smoother, more controllable latent interpolation.
- GAN may not converge to a stable solution (requires more tuning or careful hyperparameter selection and implementation).


## Stage 8 - Final analysis and conclusions

This stage is intentionally split into an evidence-backed dynamic summary (computed from experiment outputs) and short qualitative limitations. The conclusion above references the measured reconstruction comparison, latent-dimension sweep, denoising comparison, and GAN/VAE behavior to answer the final control questions directly.

## Extra stage - denoising variants (AE and VAE)

This block trains denoising models with noisy inputs and clean targets, then compares them with the baseline augmented-only AE/VAE.

In [29]:
class DenoiseWrapper(torch.utils.data.Dataset):
    def __init__(self, base_ds, noise_std: float = 0.25):
        self.base_ds = base_ds
        self.noise_std = noise_std

    def __len__(self):
        return len(self.base_ds)

    def __getitem__(self, idx):
        clean, y = self.base_ds[idx]
        noisy = torch.clamp(clean + self.noise_std * torch.randn_like(clean), 0, 1)
        return noisy, clean, y


def make_denoise_loaders(batch_size: int, noise_std: float = 0.25, train_limit: int | None = 20000, test_limit: int | None = 5000):
    train_idx = np.arange(len(train_ds)) if train_limit is None else np.arange(train_limit)
    test_idx = np.arange(len(test_ds)) if test_limit is None else np.arange(test_limit)
    tr = DenoiseWrapper(Subset(train_ds, train_idx), noise_std=noise_std)
    te = DenoiseWrapper(Subset(test_ds, test_idx), noise_std=noise_std)
    train_loader = DataLoader(tr, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader = DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=0)
    return train_loader, test_loader

In [30]:
def train_denoise_ae(cfg: dict[str, Any]) -> dict[str, Any]:
    tc_kwargs = {k: cfg[k] for k in TrainConfig.__dataclass_fields__.keys() if k in cfg}
    tc = TrainConfig(**tc_kwargs)
    noise_std = float(cfg.get("noise_std", 0.25))
    train_loader, test_loader = make_denoise_loaders(tc.batch_size, noise_std=noise_std)
    model = ConvAE(latent_dim=tc.latent_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=tc.lr)

    train_losses, val_losses = [], []
    for _ in tqdm(range(tc.epochs), desc="DAE epochs", leave=False):
        model.train()
        tr_loss = 0.0
        for x_noisy, x_clean, _ in train_loader:
            x_noisy = x_noisy.to(device)
            x_clean = x_clean.to(device)
            opt.zero_grad()
            x_hat, _ = model(x_noisy)
            loss = F.binary_cross_entropy(x_hat, x_clean)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * x_noisy.size(0)
        train_losses.append(tr_loss / len(train_loader.dataset))

        model.eval()
        vl_loss = 0.0
        with torch.no_grad():
            for x_noisy, x_clean, _ in test_loader:
                x_noisy = x_noisy.to(device)
                x_clean = x_clean.to(device)
                x_hat, _ = model(x_noisy)
                vl_loss += F.binary_cross_entropy(x_hat, x_clean).item() * x_noisy.size(0)
        val_losses.append(vl_loss / len(test_loader.dataset))

    with torch.no_grad():
        x_noisy, x_clean, y_batch = next(iter(test_loader))
        x_noisy = x_noisy.to(device)
        x_hat, z = model(x_noisy)

    return {
        "cfg": cfg,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_bce": float(val_losses[-1]),
        "noisy": x_noisy.detach().cpu().numpy(),
        "clean": x_clean.numpy(),
        "recon": x_hat.detach().cpu().numpy(),
        "labels": y_batch.numpy(),
        "latent": z.detach().cpu().numpy(),
        "model_state": cpu_state_dict(model.state_dict()),
    }


def train_denoise_vae(cfg: dict[str, Any]) -> dict[str, Any]:
    tc_kwargs = {k: cfg[k] for k in TrainConfig.__dataclass_fields__.keys() if k in cfg}
    tc = TrainConfig(**tc_kwargs)
    noise_std = float(cfg.get("noise_std", 0.25))
    train_loader, test_loader = make_denoise_loaders(tc.batch_size, noise_std=noise_std)
    model = ConvVAE(latent_dim=tc.latent_dim).to(device)
    opt = optim.Adam(model.parameters(), lr=tc.lr)

    history = {"train_total": [], "train_rec": [], "train_kl": [], "val_total": []}
    for _ in tqdm(range(tc.epochs), desc="DVAE epochs", leave=False):
        model.train()
        t_total = t_rec = t_kl = 0.0
        for x_noisy, x_clean, _ in train_loader:
            x_noisy = x_noisy.to(device)
            x_clean = x_clean.to(device)
            opt.zero_grad()
            x_hat, mu, logvar, _ = model(x_noisy)
            rec = F.binary_cross_entropy(x_hat, x_clean, reduction="sum") / x_noisy.size(0)
            kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x_noisy.size(0)
            loss = rec + tc.beta * kl
            loss.backward()
            opt.step()
            t_total += loss.item() * x_noisy.size(0)
            t_rec += rec.item() * x_noisy.size(0)
            t_kl += kl.item() * x_noisy.size(0)

        history["train_total"].append(t_total / len(train_loader.dataset))
        history["train_rec"].append(t_rec / len(train_loader.dataset))
        history["train_kl"].append(t_kl / len(train_loader.dataset))

        model.eval()
        v_total = 0.0
        with torch.no_grad():
            for x_noisy, x_clean, _ in test_loader:
                x_noisy = x_noisy.to(device)
                x_clean = x_clean.to(device)
                x_hat, mu, logvar, _ = model(x_noisy)
                rec = F.binary_cross_entropy(x_hat, x_clean, reduction="sum") / x_noisy.size(0)
                kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x_noisy.size(0)
                v_total += (rec + tc.beta * kl).item() * x_noisy.size(0)
        history["val_total"].append(v_total / len(test_loader.dataset))

    with torch.no_grad():
        x_noisy, x_clean, y_batch = next(iter(test_loader))
        x_noisy = x_noisy.to(device)
        x_hat, mu, logvar, z = model(x_noisy)

    return {
        "cfg": cfg,
        "history": history,
        "test_total": float(history["val_total"][-1]),
        "noisy": x_noisy.detach().cpu().numpy(),
        "clean": x_clean.numpy(),
        "recon": x_hat.detach().cpu().numpy(),
        "labels": y_batch.numpy(),
        "mu": mu.detach().cpu().numpy(),
        "logvar": logvar.detach().cpu().numpy(),
        "latent": z.detach().cpu().numpy(),
        "model_state": cpu_state_dict(model.state_dict()),
    }

In [31]:
dae_grid_df, dae_best_cfg, dae_res = run_grid_search("DAE", HEAVY_GRID_CONFIGS["dae"], trainer=train_denoise_ae)
dvae_grid_df, dvae_best_cfg, dvae_res = run_grid_search("DVAE", HEAVY_GRID_CONFIGS["dvae"], trainer=train_denoise_vae)
display(dae_grid_df.head(10))
display(dvae_grid_df.head(10))
print("Best DAE config:", dae_best_cfg)
print("Best DVAE config:", dvae_best_cfg)

denoise_cmp_ae = pd.DataFrame([
    {"model": "AE (base)", "metric": float(ae_res["test_bce"]), "kind": "baseline"},
    {"model": "AE (denoise)", "metric": float(dae_res["test_bce"]), "kind": "denoise"},
])
denoise_cmp_vae = pd.DataFrame([
    {"model": "VAE (base)", "metric": float(vae_res["test_total"]), "kind": "baseline"},
    {"model": "VAE (denoise)", "metric": float(dvae_res["test_total"]), "kind": "denoise"},
])
fig_dn_cmp_ae = px.bar(denoise_cmp_ae, x="model", y="metric", color="kind", title="AE: baseline vs denoising")
style_fig(fig_dn_cmp_ae, "AE vs DAE (validation metric)").show()
fig_dn_cmp_vae = px.bar(denoise_cmp_vae, x="model", y="metric", color="kind", title="VAE: baseline vs denoising")
style_fig(fig_dn_cmp_vae, "VAE vs DVAE (validation objective)").show()
denoise_cmp = pd.concat([
    denoise_cmp_ae.assign(family="AE"),
    denoise_cmp_vae.assign(family="VAE"),
], ignore_index=True)

stack_dae = np.concatenate([
    dae_res["noisy"].squeeze(1)[:10],
    dae_res["clean"].squeeze(1)[:10],
    dae_res["recon"].squeeze(1)[:10],
], axis=0)
fig_dae = show_image_grid(stack_dae, "DAE rows: noisy / clean / reconstruction", ncols=10)
fig_dae.show()

stack_dvae = np.concatenate([
    dvae_res["noisy"].squeeze(1)[:10],
    dvae_res["clean"].squeeze(1)[:10],
    dvae_res["recon"].squeeze(1)[:10],
], axis=0)
fig_dvae = show_image_grid(stack_dvae, "DVAE rows: noisy / clean / reconstruction", ncols=10)
fig_dvae.show()

DVAE grid: 100%|██████████| 48/48 [00:00<00:00, 60.19it/s, best_so_far=151.1775, loaded=1]


,model,metric,metric_key,cache_path,loaded,epochs,lr,latent_dim,batch_size,beta,noise_std
0,DAE,0.094123,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,1.0,0.20
1,DAE,0.095597,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,1.0,0.20
2,DAE,0.096637,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,1.0,0.20
3,DAE,0.098991,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,32,256,1.0,0.20
4,DAE,0.103119,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,1.0,0.35
5,DAE,0.104249,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,1.0,0.35
6,DAE,0.106920,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,1.0,0.35
7,DAE,0.108717,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,32,256,1.0,0.35
8,DAE,0.114232,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,16,256,1.0,0.20
9,DAE,0.116362,test_bce,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,16,256,1.0,0.20


,model,metric,metric_key,cache_path,loaded,epochs,lr,latent_dim,batch_size,beta,noise_std
0,DVAE,104.745453,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,0.5,0.20
1,DVAE,106.008029,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,0.5,0.20
2,DVAE,106.359201,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,0.5,0.20
3,DVAE,107.908732,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0005,32,256,0.5,0.20
4,DVAE,110.428815,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,32,256,0.5,0.35
5,DVAE,111.280923,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,32,256,0.5,0.35
6,DVAE,111.441035,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,32,256,0.5,0.35
7,DVAE,111.786825,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0010,16,256,0.5,0.20
8,DVAE,113.741067,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,40,0.0010,16,256,0.5,0.20
9,DVAE,114.093353,test_total,/Users/lopatenko/Desktop/itmo/itmo-ml-2025/dl/...,True,60,0.0005,16,256,0.5,0.20


Best DAE config: {'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 1.0, 'noise_std': 0.2}
Best DVAE config: {'epochs': 60, 'lr': 0.001, 'latent_dim': 32, 'batch_size': 256, 'beta': 0.5, 'noise_std': 0.2}


Denoising comparison notes: heavy-grid denoising variants are optimized for recovering clean targets from noisy inputs, and the selected best configs from DAE/DVAE search should be read together with clean-input baselines. This makes robustness conclusions evidence-based across both conditions.

In [32]:
best_recon = comparison_df.sort_values('bce_recon').iloc[0]
best_dn_ae = denoise_cmp_ae.sort_values('metric').iloc[0]
best_dn_vae = denoise_cmp_vae.sort_values('metric').iloc[0]
best_ld = ld_df.sort_values('metric').iloc[0]

display(Markdown(f"""
## Evidence-backed final summary

- **Reconstruction winner (shared BCE comparison):** {best_recon['model']} with `{float(best_recon['bce_recon']):.4f}`.
- **Best latent-dimension setting in sweep:** `latent_dim={int(best_ld['latent_dim'])}` with objective `{float(best_ld['metric']):.4f}`.
- **AE denoising (lower val BCE is better):** {best_dn_ae['model']} with `{float(best_dn_ae['metric']):.4f}`.
- **VAE denoising (lower val objective is better):** {best_dn_vae['model']} with `{float(best_dn_vae['metric']):.4f}`.
- **GAN vs VAE trade-off:** GAN samples are disapointingly strange and unstable, while VAE keeps more stable latent interpolation and controllability.
"""))


## Evidence-backed final summary

- **Reconstruction winner (shared BCE comparison):** AE with `0.0858`.
- **Best latent-dimension setting in sweep:** `latent_dim=32` with objective `104.2820`.
- **AE denoising (lower val BCE is better):** AE (base) with `0.0898`.
- **VAE denoising (lower val objective is better):** VAE (base) with `101.4468`.
- **GAN vs VAE trade-off:** GAN samples are disapointingly strange and unstable, while VAE keeps more stable latent interpolation and controllability.


In [33]:
from datetime import datetime
from dateutil import tz

current_time = datetime.now(tz.gettz('Etc/GMT-3'))
print(f"Current time in UTC+3: {current_time}")

Current time in UTC+3: 2026-05-02 14:05:31.982581+03:00


**This research was completed as part of the "Deep Learning" course at ITMO University, 2026**